In [1]:
from tqdm import tqdm
import logging
import sys
import config
from utils.data_process_helper import load_parquet_data
from frame_classifier.mtg_card_frame_classifier import MTGCardFrameClassifier

In [2]:
logging.basicConfig(
    level="INFO",
    format="%(asctime)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)]
)
logger = logging.getLogger(__name__)

In [3]:
data = load_parquet_data(config.VALIDATION_DATA_FILE_PATH, config.VALIDATION_IMAGE_PATH)

2025-10-04 23:16:47 - INFO - Loading parquet file: _data/validation/validation_data.parquet
2025-10-04 23:16:47 - INFO - Loaded 5315 rows
2025-10-04 23:16:47 - INFO - Columns: ['layout', 'name', 'setCode', 'number', 'isOldSet']
2025-10-04 23:16:47 - INFO - Created 5050 annotations
2025-10-04 23:16:47 - INFO - ✗ Missing 0 images


### Validate frame classifier

In [4]:
def validate(classifier: MTGCardFrameClassifier):
    correctness = 0.0
    for img_path, label in tqdm(data.items()):
        try:
            prediction = classifier.predict(config.VALIDATION_IMAGE_PATH / img_path)
            if prediction == label.isOldSet:
                correctness+=1
        except Exception as e:
            logger.error(f"Fail to predict image {img_path}. Error: {e}")
    logger.info(correctness / len(data))

In [5]:
classifier = MTGCardFrameClassifier(model_path= config.CARD_FRAME_CLASSIFIER_MODEL_PATH / "best_model.pth")

In [6]:
validate(classifier)

100%|██████████| 5050/5050 [01:11<00:00, 70.36it/s]

2025-10-04 23:18:10 - INFO - 0.9996039603960396


#### Distillation Model

In [7]:
classifier = MTGCardFrameClassifier(model_path= config.CARD_FRAME_CLASSIFIER_MODEL_PATH / "distillation_best.pth", distilled=True)

In [8]:
validate(classifier)

100%|██████████| 5050/5050 [00:51<00:00, 98.80it/s] 

2025-10-04 23:19:01 - INFO - 0.9964356435643564


### Validate Region Detection